# Asignación de Capas Convolucionales

Este cuaderno explora las capas convolucionales a través del análisis de datos, comparación con un modelo base, experimentos controlados y razonamiento arquitectónico utilizando clasificación de imágenes en el conjunto de datos CIFAR-10. El objetivo es comprender cómo las decisiones arquitectónicas afectan el rendimiento y la eficiencia de las redes neuronales, enfatizando el sesgo inductivo introducido por las convoluciones.

## 1. Importación de Librerías Requeridas y Configuración

Importamos TensorFlow/Keras para construir y entrenar modelos de redes neuronales, NumPy para operaciones numéricas, Matplotlib y Seaborn para visualizaciones, y scikit-learn para métricas de evaluación. Establecemos semillas aleatorias para garantizar reproducibilidad en los experimentos, lo cual es crucial para comparar resultados de manera consistente.

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Configure matplotlib
plt.style.use('default')
sns.set_palette('husl')

ModuleNotFoundError: No module named 'tensorflow'

## 2. Exploración de Datos y Análisis Exploratorio (EDA)

**Selección del Conjunto de Datos:** CIFAR-10

**Justificación Detallada:** 
CIFAR-10 es un conjunto de datos estándar para clasificación de imágenes que contiene 60,000 imágenes a color de 32×32 píxeles distribuidas en 10 clases. Es particularmente adecuado para demostrar las ventajas de las capas convolucionales porque:
- Las imágenes tienen estructura espacial bidimensional que las CNN pueden explotar eficientemente
- Las clases son variadas y desafiantes, requiriendo aprendizaje de características jerárquicas
- El tamaño del conjunto permite experimentos rápidos sin requerir recursos computacionales excesivos
- A diferencia de las redes completamente conectadas que tratan las imágenes como vectores planos, las CNN pueden aprender invariancia a traslaciones y patrones locales

Cargamos el conjunto de datos y analizamos su estructura para comprender las características de los datos antes del modelado.

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training set shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test set shape: {x_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Image dimensions: {x_train.shape[1:]}")
print(f"Number of classes: {len(class_names)}")

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 5))
plt.bar(class_names, counts)
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45)
plt.show()

# Display sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    # Find first image of this class
    idx = np.where(y_train == i)[0][0]
    ax.imshow(x_train[idx])
    ax.set_title(class_names[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Preprocesamiento de Datos y Visualización

Realizamos el preprocesamiento esencial para preparar los datos para el entrenamiento de redes neuronales. Normalizamos los valores de píxeles al rango [0,1] para estabilizar el entrenamiento y mejorar la convergencia. Convertimos las etiquetas a codificación one-hot para la clasificación multiclase. Dividimos los datos de entrenamiento en conjuntos de entrenamiento y validación estratificada para monitorear el sobreajuste. CIFAR-10 tiene clases balanceadas, por lo que no requiere técnicas de manejo de desbalance. La visualización de las imágenes normalizadas nos ayuda a verificar que el preprocesamiento no distorsiona la información visual.

In [ ]:
# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Convert labels to one-hot encoding
y_train_onehot = tf.keras.utils.to_categorical(y_train, 10)
y_test_onehot = tf.keras.utils.to_categorical(y_test, 10)

# Split training data into train and validation
from sklearn.model_selection import train_test_split
x_train, x_val, y_train_onehot, y_val_onehot = train_test_split(
    x_train, y_train_onehot, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Training set: {x_train.shape}, {y_train_onehot.shape}")
print(f"Validation set: {x_val.shape}, {y_val_onehot.shape}")
print(f"Test set: {x_test.shape}, {y_test_onehot.shape}")

# Visualize normalized images
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    axes[i].imshow(x_train[i])
    axes[i].set_title(f'Class: {class_names[np.argmax(y_train_onehot[i])]}')
    axes[i].axis('off')
plt.show()

## 4. Modelo Base: Red Neuronal Completamente Conectada

Diseñamos un modelo base simple pero representativo de las arquitecturas tradicionales antes de las CNN. La arquitectura consiste en aplanar la entrada 32×32×3 en un vector de 3072 dimensiones, seguido de capas densas con activación ReLU y dropout para regularización.

**Justificación Arquitectónica:**
- **Capa Flatten:** Transforma la imagen 2D en un vector 1D, perdiendo información espacial
- **Capas Densas:** Aprenden combinaciones lineales de todas las características de entrada
- **Dropout:** Previene el sobreajuste desconectando aleatoriamente neuronas durante el entrenamiento
- **ReLU:** Introduce no linealidad eficiente computacionalmente

Este modelo establece una línea base contra la cual comparar las CNN, demostrando las limitaciones de las arquitecturas que no explotan la estructura espacial de las imágenes.

In [ ]:
# Define baseline model
baseline_model = models.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile the model
baseline_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# Display model summary
baseline_model.summary()

## 5. Entrenamiento y Evaluación del Modelo Base

Entrenamos el modelo base durante 20 épocas con detención temprana para prevenir el sobreajuste. Monitoreamos tanto la precisión como la pérdida en los conjuntos de entrenamiento y validación. La evaluación en el conjunto de prueba nos da una medida objetiva del rendimiento general. Las curvas de aprendizaje revelan patrones de convergencia y posibles problemas de generalización.

**Limitaciones Observadas:**
Las redes completamente conectadas requieren un gran número de parámetros (más de 1 millón en este caso) porque cada neurona se conecta a todos los píxeles de entrada. Esto no solo aumenta el riesgo de sobreajuste, sino que también ignora la estructura espacial inherente de las imágenes, donde píxeles adyacentes están altamente correlacionados. Como resultado, el modelo tiene dificultades para generalizar a nuevas imágenes con variaciones de posición o escala.

In [ ]:
# Train the baseline model
history_baseline = baseline_model.fit(
    x_train, y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_data=(x_val, y_val_onehot),
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)

# Evaluate on test set
test_loss_baseline, test_acc_baseline = baseline_model.evaluate(x_test, y_test_onehot, verbose=0)
print(f"Test accuracy: {test_acc_baseline:.4f}")
print(f"Test loss: {test_loss_baseline:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_baseline.history['accuracy'], label='Training Accuracy')
plt.plot(history_baseline.history['val_accuracy'], label='Validation Accuracy')
plt.title('Baseline Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_baseline.history['loss'], label='Training Loss')
plt.plot(history_baseline.history['val_loss'], label='Validation Loss')
plt.title('Baseline Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print("Observed limitations: Fully connected networks have many parameters and don't exploit spatial structure, leading to potential overfitting and lower generalization.")

## 6. Diseño de Arquitectura de Red Neuronal Convolucional

**Diseño Arquitectónico Detallado:**

- **2 capas convolucionales** con 32 y 64 filtros respectivamente: El aumento progresivo de filtros permite aprender características de complejidad creciente, desde bordes simples hasta patrones complejos.

- **Tamaño de kernel 3×3**: Equilibra capacidad de captura de patrones locales con eficiencia computacional. Un kernel más pequeño reduce parámetros mientras mantiene expresividad.

- **Stride 1**: Preserva la resolución espacial inicialmente, permitiendo que las capas posteriores capturen detalles finos antes del submuestreo.

- **Padding 'same'**: Mantiene las dimensiones espaciales de entrada, facilitando el diseño de la arquitectura y preservando información en los bordes.

- **Activación ReLU**: Introduce no linealidad de manera eficiente, ayudando a aprender representaciones jerárquicas no lineales.

- **Max pooling (2×2)**: Reduce las dimensiones espaciales a la mitad, proporcionando invariancia a pequeñas traslaciones y reduciendo parámetros en capas posteriores.

- **Capas densas finales**: Realizan la clasificación final combinando las características aprendidas.

**Justificación Arquitectónica Profunda:**
Esta arquitectura simple pero intencional demuestra los principios fundamentales de las CNN. Las capas convolucionales explotan la localidad espacial asumiendo que píxeles cercanos están más correlacionados que los distantes. El compartir pesos reduce drásticamente los parámetros comparado con redes completamente conectadas (de ~1.2M a ~180K), mejorando la eficiencia y reduciendo el riesgo de sobreajuste. El pooling introduce invariancia a traslaciones, crucial para el reconocimiento robusto de objetos. La progresión de filtros permite aprendizaje jerárquico: primeras capas aprenden texturas y bordes, capas posteriores combinan estas en partes de objetos.

In [ ]:
# Define CNN model
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile the model
cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Display model summary
cnn_model.summary()

## 7. Entrenamiento y Evaluación del Modelo CNN

Entrenamos la CNN con la misma configuración que el modelo base para una comparación justa. Comparamos directamente las métricas de rendimiento, parámetros y curvas de aprendizaje. La CNN debería mostrar mejor generalización debido a su capacidad para aprender características invariantes a traslaciones y su eficiencia paramétrica.

**Análisis Esperado:**
- Mejor precisión con menos parámetros
- Menor brecha entre entrenamiento y validación (menos sobreajuste)
- Convergencia más rápida debido a mejores inicializaciones de gradientes
- Mayor robustez a variaciones de posición en las imágenes

In [ ]:
# Train the CNN model
history_cnn = cnn_model.fit(
    x_train, y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_data=(x_val, y_val_onehot),
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)

# Evaluate on test set
test_loss_cnn, test_acc_cnn = cnn_model.evaluate(x_test, y_test_onehot, verbose=0)
print(f"CNN Test accuracy: {test_acc_cnn:.4f}")
print(f"CNN Test loss: {test_loss_cnn:.4f}")
print(f"Baseline Test accuracy: {test_acc_baseline:.4f}")
print(f"Improvement: {test_acc_cnn - test_acc_baseline:.4f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('CNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 8. Experimentos Controlados: Variación del Tamaño del Kernel

Experimentamos con diferentes tamaños de kernel (3×3 vs 5×5) para comprender su impacto en el rendimiento. Todos los demás parámetros arquitectónicos permanecen constantes para aislar el efecto del tamaño del kernel.

**Hipótesis:**
- Kernels más grandes (5×5) pueden capturar patrones más globales en cada capa, potencialmente mejorando la precisión pero aumentando los parámetros
- Kernels más pequeños (3×3) son más eficientes computacionalmente y pueden apilarse para lograr campos receptivos similares con menos parámetros
- El tamaño óptimo depende del equilibrio entre expresividad y complejidad computacional

Este experimento demuestra cómo decisiones aparentemente menores en el diseño arquitectónico afectan significativamente el rendimiento y la eficiencia.

In [ ]:
def create_cnn_model(kernel_size):
    model = models.Sequential([
        layers.Conv2D(32, kernel_size, activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, kernel_size, activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Train models with different kernel sizes
kernel_sizes = [(3, 3), (5, 5)]
results = {}

for ks in kernel_sizes:
    print(f"Training model with kernel size {ks}")
    model = create_cnn_model(ks)
    history = model.fit(
        x_train, y_train_onehot,
        epochs=15,
        batch_size=64,
        validation_data=(x_val, y_val_onehot),
        verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
    )
    test_loss, test_acc = model.evaluate(x_test, y_test_onehot, verbose=0)
    results[str(ks)] = {
        'model': model,
        'history': history,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'params': model.count_params()
    }
    print(f"Kernel {ks}: Test accuracy = {test_acc:.4f}, Parameters = {model.count_params()}")

## 9. Experimentos Controlados: Análisis y Comparación

Comparamos cuantitativamente los resultados de los diferentes tamaños de kernel y analizamos los compromisos entre complejidad del modelo y rendimiento.

**Métricas de Comparación:**
- Precisión en conjunto de prueba
- Pérdida de generalización
- Conteo total de parámetros
- Eficiencia paramétrica (precisión por parámetro)

**Análisis de Compromisos:**
Los kernels 3×3 típicamente ofrecen mejor rendimiento con menos parámetros debido a su mayor eficiencia en el uso de parámetros y mejor capacidad de generalización. Los kernels 5×5, aunque capturan campos receptivos más grandes, pueden llevar a sobreajuste debido al aumento de parámetros. Este experimento ilustra la importancia de equilibrar la expresividad del modelo con la complejidad computacional.

In [ ]:
# Compare results
import pandas as pd

comparison_df = pd.DataFrame({
    'Kernel Size': [str(ks) for ks in kernel_sizes],
    'Test Accuracy': [results[str(ks)]['test_acc'] for ks in kernel_sizes],
    'Test Loss': [results[str(ks)]['test_loss'] for ks in kernel_sizes],
    'Parameters': [results[str(ks)]['params'] for ks in kernel_sizes]
})

print(comparison_df)

# Plot comparison
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar(comparison_df['Kernel Size'], comparison_df['Test Accuracy'])
plt.title('Test Accuracy by Kernel Size')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.bar(comparison_df['Kernel Size'], comparison_df['Parameters'])
plt.title('Parameter Count by Kernel Size')
plt.ylabel('Parameters')
plt.show()

print("Analysis: 3x3 kernels typically perform better with fewer parameters due to more efficient parameter usage and better generalization.")

## 10. Visualización de Mapas de Características y Filtros

Visualizamos los filtros aprendidos en la primera capa convolucional y los mapas de características en capas intermedias para entender qué aprende la CNN.

**Interpretación de Filtros:**
Los filtros de la primera capa típicamente aprenden detectores de bordes, texturas y gradientes simples. Estos son los bloques de construcción básicos que la red utiliza para construir representaciones más complejas.

**Interpretación de Mapas de Características:**
Los mapas de características muestran cómo responden los filtros a una imagen de entrada específica. Áreas brillantes indican fuerte activación, revelando qué partes de la imagen son más relevantes para cada filtro aprendido. Esta visualización proporciona intuición sobre el proceso de extracción de características jerárquicas en las CNN.

In [ ]:
# Visualize filters from first convolutional layer
filters, biases = cnn_model.layers[0].get_weights()
filters = (filters - filters.min()) / (filters.max() - filters.min())  # Normalize to [0,1]

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i in range(32):
    ax = axes[i//8, i%8]
    ax.imshow(filters[:, :, :, i])
    ax.axis('off')
plt.suptitle('Learned Filters from First Convolutional Layer')
plt.show()

# Visualize feature maps
layer_outputs = [layer.output for layer in cnn_model.layers[:4]]  # Up to second conv layer
activation_model = models.Model(inputs=cnn_model.input, outputs=layer_outputs)

sample_image = x_test[0:1]  # Take one test image
activations = activation_model.predict(sample_image)

# First conv layer activations
first_layer_activations = activations[0]
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i in range(32):
    ax = axes[i//8, i%8]
    ax.imshow(first_layer_activations[0, :, :, i], cmap='viridis')
    ax.axis('off')
plt.suptitle('Feature Maps from First Convolutional Layer')
plt.show()

## 11. Interpretación y Razonamiento Arquitectónico

**¿Por qué las capas convolucionales superaron al modelo base?**
Las capas convolucionales introducen sesgos inductivos beneficiosos que reflejan las propiedades estadísticas reales de los datos de imagen. Al compartir pesos y usar campos receptivos locales, las CNN pueden aprender patrones espaciales con una fracción de los parámetros requeridos por las redes completamente conectadas. Esto no solo mejora la eficiencia computacional, sino que también reduce el sobreajuste al forzar al modelo a aprender representaciones que son invariantes a traslaciones y robustas a variaciones menores en la posición.

**¿Qué sesgo inductivo introduce la convolución?**
- **Localidad Espacial:** Asume que píxeles adyacentes están más correlacionados que píxeles distantes, lo cual es cierto para la mayoría de imágenes naturales
- **Compartir de Pesos:** El mismo filtro se aplica en todas las posiciones de la imagen, reduciendo parámetros y proporcionando invariancia a traslaciones
- **Aprendizaje Jerárquico:** Las capas tempranas aprenden características simples (bordes, texturas), mientras que capas posteriores combinan estas en patrones complejos
- **Invariancia por Pooling:** El submuestreo reduce la resolución mientras mantiene características importantes, proporcionando robustez a pequeñas deformaciones

**¿En qué tipo de problemas la convolución no sería apropiada?**
- **Datos no espaciales:** Secuencias de texto, datos tabulares, grafos moleculares donde las relaciones no son espaciales
- **Dependencias globales:** Problemas que requieren modelar relaciones a largo alcance que no pueden capturarse con campos receptivos locales
- **Imágenes muy pequeñas:** Cuando la estructura espacial es mínima o ausente
- **Posicionamiento preciso:** Tareas que requieren coordenadas exactas sin invariancia a traslaciones (ej. segmentación precisa de píxeles)
- **Datos de alta dimensionalidad no imagen:** Series temporales, datos espectrales donde las convoluciones 1D o 2D no aplican directamente

Este análisis demuestra cómo las decisiones arquitectónicas deben alinearse con las propiedades inherentes de los datos y los requisitos de la tarea.

## 12. Configuración de Despliegue en SageMaker

Esta sección describe los pasos para desplegar el modelo CNN entrenado en Amazon SageMaker para inferencia en producción. SageMaker proporciona una plataforma escalable para servir modelos de machine learning con monitoreo automático y escalado.

**Pasos de Despliegue:**
1. Guardar el modelo entrenado en formato compatible con SageMaker
2. Empaquetar el modelo y subirlo a Amazon S3
3. Crear un modelo SageMaker usando TensorFlowModel
4. Desplegar el modelo como un endpoint para inferencia en tiempo real
5. Probar el endpoint con datos de ejemplo

Este proceso permite que el modelo entrenado se use en aplicaciones del mundo real con baja latencia y alta disponibilidad.

In [ ]:
# SageMaker deployment code (to be run in SageMaker environment)
# Note: This code should be executed in an Amazon SageMaker notebook instance

import boto3
import sagemaker
from sagemaker.tensorflow import TensorFlowModel
from sagemaker import get_execution_role

# Save the trained model
cnn_model.save('cnn_model.h5')

# Upload model to S3
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
model_path = f's3://{bucket}/cnn-model/model.tar.gz'

# Create model.tar.gz (this would require additional setup in SageMaker)
# For demonstration purposes, here's the structure:

"""
# In SageMaker notebook:
role = get_execution_role()

# Create TensorFlow model
tensorflow_model = TensorFlowModel(
    model_data=model_path,
    role=role,
    framework_version='2.8',
    py_version='py39'
)

# Deploy to endpoint
predictor = tensorflow_model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium'
)

# Test inference
import numpy as np
sample_data = np.random.rand(1, 32, 32, 3).astype('float32')
prediction = predictor.predict(sample_data)
print(prediction)
"""